Folder Hierarchy Analysis Tool

This script recursively analyzes a directory structure and generates a detailed DataFrame containing:
- Complete folder paths and names
- Unique hierarchical IDs (1.2.3 format)
- Parent folder information
- Depth information

Features:
- Recursively walks directory trees
- Builds a DataFrame with folder hierarchy
- Assigns unique hierarchical identifiers (1.2.4-style)

Output Columns:
- folder_path: Full path of the folder
- folder_name: Last part of the path (folder's name)
- unique_id: Hierarchical ID (e.g., 1.2.1)
- depth_from_root: Levels below the given root
- parent_folder: Name of the immediate parent folder
- parent_id: Hierarchical ID of the parent folder

Usage in Jupyter:
    root_folder = "/path/to/your/folder"
    df = build_folder_hierarchy(root_folder)
    df.head()

Requires:
    - pathlib
    - pandas
    - collections (defaultdict)

In [13]:
import os
from pathlib import Path
import pandas as pd
from collections import defaultdict

In [ ]:
def build_folder_hierarchy(root_path, max_depth=5):
    root = Path(root_path).resolve()
    # Modify the folder collection to check depth during collection
    all_folders = []
    for p in root.rglob("*"):
        if p.is_dir():
            # Calculate depth from root
            try:
                depth = len(p.relative_to(root).parts)
                if depth <= max_depth:
                    all_folders.append(p)
            except ValueError:
                continue
    
    all_folders = sorted(all_folders)
    all_folders.insert(0, root)  # include the root
    
    # Track folder IDs and sibling counters
    folder_id_map = {}  # maps folder path → hierarchical ID 
    child_counters = defaultdict(int)  # maps parent path → child index
    
    # Track records for output
    records = []
    
    # Assign IDs in traversal order
    for folder in all_folders:
        folder_str = str(folder)
        parent = folder.parent
        parent_str = str(parent)
        
        # Root folder: assign top-level ID
        if folder == root:
            folder_id = "1"
            parent_folder = ""
            parent_id = ""
        else:
            parent_id = folder_id_map.get(parent_str, "X")  # fallback if parent not yet assigned
            child_counters[parent_str] += 1
            index = child_counters[parent_str]
            folder_id = f"{parent_id}.{index}"
            parent_folder = parent.name
            
        folder_id_map[folder_str] = folder_id
        
        records.append({
            "folder_path": folder_str,
            "folder_name": folder.name,
            "unique_id": folder_id,
            "depth_from_root": len(folder.relative_to(root).parts),
            "parent_folder": parent_folder if folder != root else "",
            "parent_id": parent_id
        })
            
    return pd.DataFrame(records)


In [ ]:
# Specify your root folder path directly
root_folder = "TestFolder"  # Replace with your actual path
# C:\Users\rhys.wells\Desktop\Inbox\DE_Tools\Explorations\Other\Folder-Tools\Cataloging\folder-cataloging.ipynb
# root_folder="R:\Nevis\15. Technical Services (Site Operating Instructions)\MEASUREMENT ARRANGEMENT\PowerBI"

# root_folder = r"R:\Nevis"

# Run the hierarchy analysis
max_depth=3
df = build_folder_hierarchy(root_folder,max_depth)

# Save to CSV
output_file = "Nevis_folder_hierarchy.csv"
df.to_csv(output_file, index=False)
print(f"Hierarchy written to {output_file}")

KeyboardInterrupt: 

In [17]:
# Display first few rows of the DataFrame
df.head(20)


,folder_path,folder_name,unique_id,depth_from_root,parent_folder,parent_id
0,\\prdawsfapnev01\shared\Nevis\15. Technical Se...,PowerBI,1,0,,
1,\\prdawsfapnev01\shared\Nevis\15. Technical Se...,Archive,1.1,1,PowerBI,1
2,\\prdawsfapnev01\shared\Nevis\15. Technical Se...,Supplements,1.2,1,PowerBI,1
